# LFM Segmentation Example Workflow
This notebook is an example workflow of doing binary segmentation on visual light (RGB) bands of Lunar data. 

You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/finetune_dinov3.ipynb
```

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

## Imports, Dino Repo Clone

In [1]:
import os
import sys
import torch
import subprocess
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

%matplotlib inline

In [2]:
repo_dir = "lfm"

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"])
# else:
#     subprocess.run(["git", "-C", repo_dir, "pull"])
subprocess.run(["git", "-C", repo_dir, "checkout", "develop"])
subprocess.run(["git", "-C", repo_dir, "pull"])

Already on 'develop'


Your branch is up to date with 'origin/develop'.


From https://github.com/nasa-nccs-hpda/lfm
   255a31b..bb6c128  develop    -> origin/develop


Updating 255a31b..bb6c128
Fast-forward
 lfm/tasks/inst_segmentation/mask2former_model.py   |  77 ++++++--
 .../finetune_dinov3_inst_seg_mask2former.ipynb     | 204 +++++++++++++++++----
 2 files changed, 229 insertions(+), 52 deletions(-)


CompletedProcess(args=['git', '-C', 'lfm', 'pull'], returncode=0)

In [3]:
sys.path.append("lfm")

from lfm.tasks.inst_segmentation.model import DINOInstanceSegmentation, load_dinov3_encoder
from lfm.tasks.inst_segmentation.dataset import get_dataloaders
from lfm.tasks.inst_segmentation.driver import train_model
from lfm.tasks.inst_segmentation.mask2former_model_sat import create_mask2former_dinov3_model_sat
from lfm.tasks.inst_segmentation.mask2former_model import create_mask2former_dinov3_model
# from lfm.tasks.segmentation.utils import install_termcolor_locally

### Install termcolor package (required by DinoV3)

In [4]:
# install_termcolor_locally()

## Main workflow

1. Define user-configured variables
2. Create dataloaders from files on /explore/nobackup/.
3. Load DinoV3 encoder, create encoder/decoder finetuning model.
4. Train model, print training stats, and visualize results. 

### User Config

In [5]:
# Weights URL (received after registering for DINOV3)
weights_URL = (
    "https://dinov3.llamameta.net/dinov3_vitl16/"
    "dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
    "?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiNDloYXZtdThkZGh3eGw3aH"
    "JwNjQwa3E3IiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXR"
    "cLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE"
    "3Njc5OTI2Njl9fX1dfQ__"
    "&Signature=neHREO7xc90azhmnF3r9qPztYJ5L2wO-EZkVKh6ECzR5H2YGzdK3dcF"
    "WQISNb6xYo3R5EO39FKJ7bwELXA%7EgoBqDbk-jm-9n9%7EVxtEOmWVx73usrILMwhyi"
    "cP5-448rbnUzOEM0lPkGS829mOBJkaSxxSsbkQ0VpVBcScNEFcpaNOZ--BeHxCHdTFV"
    "NGkhlEaCYPUbYyHYbTgDQntH2AsKYJTWw4NIEZJZLX9wjCOYKQ-YG86d0HJAvsdF79X"
    "vITPgJSA0U-4Z1CzIkQhZb0N-7-XnbZmnJJi42xnNS0DsB6CTedxq0FAfiYklBY77wT"
    "JrYLba%7Epkap23ymoUTxDXA__"
    "&Key-Pair-Id=K15QRJLYKIFSLZ"
    "&Download-Request-ID=1618342689192585"
)
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'

# Data paths
INPUT_DIR = "/explore/nobackup/projects/lfm/inst_seg_viz_chips_3_band"
IMAGE_DIR = f"{INPUT_DIR}/chips"
LABEL_DIR = f"{INPUT_DIR}/labels_npy"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Location of cloned dinov3 repo
REPO_DIR = "./dinov3"

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all available samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16
NUM_EPOCHS = 10  # 10 is the default value, change to more epochs for more model learning
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
LOSS_TYPE = "instance_combined"  # Combined Dice + Binary CE loss

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Visualization and saving
CHECKPOINT_EVERY = 10  # Save checkpoint every N epochs
VISUALIZE_EVERY = 10  # Create visualizations every N epochs

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs
Using device: cuda


### Create dataloaders

In [6]:
# ============================================================================
# CREATE DATALOADERS
# ============================================================================

print("\n" + "="*60)
print("STEP 1: Creating dataloaders.")
print("="*60)

train_loader, val_loader, MEAN, STD = get_dataloaders(
    image_dir=IMAGE_DIR,
    label_dir=LABEL_DIR,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
    target_size=TARGET_SIZE,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR, 
    norm_to_one=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


STEP 1: Creating dataloaders.
Loading existing dataset statistics...
Mean per channel: [0.32744208 0.32249309 0.302985  ]
Std per channel: [0.15045737 0.15014801 0.14386101]
Limited to 500 samples
Found 500 matched image-label pairs
Dataset configured for 3 channel(s)
Train samples: 400
Val samples: 100
Train batches: 25
Val batches: 7


In [7]:
# Required for mask2former model
label2id = {
    "background": 0, 
    "crater": 1,
}
id2label = {v: k for k, v in label2id.items()}

### Load Encoder and Create Model

In [8]:
!pip install --upgrade transformers

Defaulting to user installation because normal site-packages is not writeable


In [10]:
print("\n" + "="*60)
print("Loading mask2former Dino model...")
print("="*60)

if use_sat_dino:
    
    model = create_mask2former_dinov3_model_sat(
        label2id,  
        id2label,  
        'dinov3_vitl16',
        weights_local_checkpoint,
        expected_channels=[96, 192, 384, 768],
        freeze_backbone= True,
        hub_token=None
    )
else:
    model = create_mask2former_dinov3_model(
        label2id,  
        id2label,
        expected_channels=[96, 192, 384, 768],
        freeze_backbone= True,
        hub_token=None
    )


Loading mask2former Dino model...


Loading weights:   0%|          | 0/782 [00:00<?, ?it/s]

Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-small-coco-instance
Key                    | Status   |                                                                                        
-----------------------+----------+----------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81]) vs model:torch.Size([3])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([3, 256])
class_predictor.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


AttributeError: 'DinoVisionTransformer' object has no attribute 'config'

In [ ]:
a

### Run Training

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    mode="train",
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    checkpoint_every=CHECKPOINT_EVERY,
    visualize_every=VISUALIZE_EVERY,
    loss_type=LOSS_TYPE,
    device=device
)

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:

    img = mpimg.imread(vis_filename)
    
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.axis("off")
    plt.show()